# 10 — Drawing the space, science and technology sample

**Purpose.** Select the hearings held by the congressional science committees out
of the 36,378-package corpus `04` built, and write them as their own table. This
is the demonstration subset the summarization stage runs on.

## Selection

**Both chambers, selected by committee identifier rather than by name.** The two
committees and every title they have carried:

| `authorityId` | Chamber | Name as printed | Hearings |
|---|---|---|---|
| `sscm00` | Senate | Committee on Commerce, Science, and Transportation | 1,420 |
| `hssy00` | House | Committee on Science, Space, and Technology | 586 |
| `hssy00` | House | Committee on Science and Technology | 231 |
| `hssy00` | House | Committee on Science | 146 |
| `hssy00` | *Senate* | Committee on Science | 1 |

**Senate is the larger half of this sample**, not a supplement to the House: 1,420
hearings against 964, and 65 million words against 29 million.

The House committee has been renamed twice, and all three of its titles carry
`authorityId = hssy00`, so one identifier selects the committee's whole history
where a name match would need three strings and would break on the next rename.
The last row above is the same identifier recorded against the Senate — a
metadata error in the source. Selecting on the identifier catches it; selecting on
chamber or on the name string would not.

**No Senate science committee is being left out.** A scan of every committee in
the corpus finds no separate Senate space or science committee: that jurisdiction
sits inside Commerce, so `sscm00` is the Senate in full. None of the twelve joint
committees is science-related either — they are the Helsinki Commission, the Joint
Economic Committee, the China Commission and similar. Joint *hearings* still
appear, through the identifier: 4 rows in the sample are recorded as `JOINT`.

A keyword match on `science|space|technolog` is the other obvious approach and it
is wrong in both directions: it would pull in `sp2k00`, the *Special Committee on
Year 2000 Technology Problem* (20 hearings), and it would catch the Senate row
above only by accident.

**`sscm00` is broad**, covering commerce and transportation as well as science, so
most of its hearings are not about space science. Both chambers are included as
specified, and `committee_scope` labels every row `house_science`,
`senate_commerce` or `both` so the sample can be narrowed later without rebuilding
anything.

**Every committee on the record counts, not just the first.** `04` stores
`committee_ids`, the flat list, because the scalar primary-committee field finds
940 House Science hearings where the full list finds 964 and reduces both joint
hearings to one committee.

## Usability filter

Rows must pass **both** tests to enter the sample:

* `n_chars > 8650` — the inherited threshold, a character count summed over
  paragraphs, hand-checked against transcripts that carry no conversation text.
* `has_text` — an independent signal built from the GovInfo `<TEXT NOT AVAILABLE>`
  marker, the file size, and the turn count.

The two agree on all but 64 packages of 36,378. Requiring both means either
signal is enough to call a record empty.

**Inputs.** `data/raw/02_hearings/hearings_all.parquet` — from `04`.

**Output.** `data/raw/02_hearings/hearings_sst.parquet`.

## Setup

In [1]:
import sys

sys.path.insert(0, "../lib")

import pandas as pd

from paths import HEARINGS_REPO

# The two science committees, by the identifier that survives their renames.
HOUSE_SCIENCE = "hssy00"
SENATE_COMMERCE = "sscm00"
SST_IDS = {HOUSE_SCIENCE, SENATE_COMMERCE}

SOURCE = HEARINGS_REPO / "hearings_all.parquet"
TARGET = HEARINGS_REPO / "hearings_sst.parquet"

corpus = pd.read_parquet(SOURCE)
print(f"corpus : {len(corpus):,} packages, {len(corpus.columns)} columns")
print(f"usable : {int(corpus['usable'].sum()):,}")

corpus : 36,378 packages, 40 columns
usable : 33,324


## Step 1 — Select the committees

`committee_scope` is assigned from the identifiers present on each row, so a
hearing listing both committees is labelled as such rather than assigned to
whichever appears first.

In [2]:
ids = corpus["committee_ids"].map(set)
in_house = ids.map(lambda v: HOUSE_SCIENCE in v)
in_senate = ids.map(lambda v: SENATE_COMMERCE in v)
selected = in_house | in_senate

corpus["committee_scope"] = pd.Series(
    pd.NA, index=corpus.index, dtype="object"
).mask(in_house, "house_science").mask(in_senate, "senate_commerce").mask(
    in_house & in_senate, "both"
)

print(f"House science   {HOUSE_SCIENCE}  {int(in_house.sum()):>6,}")
print(f"Senate commerce {SENATE_COMMERCE}  {int(in_senate.sum()):>6,}")
print(f"either                  {int(selected.sum()):>6,}")
print(f"both                    {int((in_house & in_senate).sum()):>6,}")
print()
# Every title the two identifiers cover, so the reader can see that the renames
# and the one mislabelled Senate row are all caught by the id.
covered = corpus.loc[selected, ["chamber", "committee_name"]].explode("committee_name")
covered = covered[covered["committee_name"].str.contains("Science|Commerce", na=False)]
print("committee titles the two identifiers cover:")
print(covered.value_counts().to_string())
print()
print("selected hearings by chamber:")
print(corpus.loc[selected, "chamber"].value_counts().to_string())

House science   hssy00     964
Senate commerce sscm00   1,420
either                   2,382
both                         2

committee titles the two identifiers cover:
chamber  committee_name                                    
SENATE   Committee on Commerce, Science, and Transportation    1414
HOUSE    Committee on Science, Space, and Technology            582
         Committee on Science and Technology                    216
         Committee on Science                                   139
JOINT    Committee on Science                                     3
HOUSE    Committee on Energy and Commerce                         2

selected hearings by chamber:
chamber
SENATE    1417
HOUSE      960
JOINT        5


## Step 2 — Apply the usability filter

Both counts are printed because the gap between them is the cost of the filter —
the number a reader needs to judge whether this sample is the committee, or the
committee minus whatever was never digitized.

In [3]:
sst = corpus.loc[selected].copy()
before = len(sst)
sst = sst.loc[sst["usable"]].copy()

dropped = corpus.loc[selected & ~corpus["usable"]]
print(f"selected by committee   {before:>6,}")
print(f"  usable                {len(sst):>6,}")
print(f"  dropped               {len(dropped):>6,}")
if len(dropped):
    reason = pd.Series("both", index=dropped.index)
    reason = reason.mask(dropped["has_text"] & (dropped["n_chars"] <= 8650), "length only")
    reason = reason.mask(~dropped["has_text"] & (dropped["n_chars"] > 8650), "has_text only")
    print("\n  why they were dropped:")
    print("   " + reason.value_counts().to_string().replace("\n", "\n   "))
    print(f"\n  their median character count: {dropped['n_chars'].median():,.0f}")
    print(f"  kept rows' median            : {sst['n_chars'].median():,.0f}")

selected by committee    2,382
  usable                 2,352
  dropped                   30

  why they were dropped:
   both             29
   has_text only     1

  their median character count: 1,036
  kept rows' median            : 232,003


## Step 3 — Write the sample

In [4]:
sst = sst.sort_values(["year", "packageId"]).reset_index(drop=True)
# Same row-group size as the corpus: the turn column is nested, and a nested
# column holding more than 2 GB in one row group cannot be read back.
sst.to_parquet(TARGET, compression="zstd", index=False, row_group_size=2000)

size = TARGET.stat().st_size
print(f"wrote {TARGET}")
print(f"  {len(sst):,} hearings, {size / 1e6:,.0f} MB zstd")
print(f"  {sst['n_words'].sum() / 1e6:,.0f} million words across "
      f"{int(sst['n_turns'].sum()):,} turns")
print(f"  fiscal years {int(sst['year'].min())}-{int(sst['year'].max())}")
print()
print(sst["committee_scope"].value_counts().to_string())

wrote /project2/njia_1089/acroh/projects/llm_method/data/raw/02_hearings/hearings_sst.parquet
  2,352 hearings, 203 MB zstd
  94 million words across 399,041 turns
  fiscal years 1996-2025

committee_scope
senate_commerce    1411
house_science       939
both                  2


## Verification

In [5]:
back = pd.read_parquet(TARGET)

assert len(back) == len(sst), f"{len(back):,} rows written against {len(sst):,}"
assert not back["packageId"].duplicated().any(), "duplicate packageId"
print(f"1) {len(back):,} hearings, packageId unique")

wrong = back.loc[~back["committee_ids"].map(lambda v: bool(SST_IDS & set(v)))]
assert len(wrong) == 0, f"{len(wrong)} rows carry neither science committee"
assert back["usable"].all(), "an unusable row entered the sample"
print("2) every row carries a science committee and passes the usability filter")

assert (back["n_turns"] > 0).all(), "a row has no turns"
assert (back["n_chars"] > 8650).all(), "a row is under the length threshold"
print(f"3) every row has turns and clears {8650:,} characters")

# Nothing that belonged in the sample was left behind.
expected = int((corpus["committee_ids"].map(lambda v: bool(SST_IDS & set(v)))
                & corpus["usable"]).sum())
assert len(back) == expected, f"{len(back):,} written, {expected:,} qualify"
print(f"4) all {expected:,} qualifying hearings are present")

print("\n5) shape of the sample:")
summary = back.groupby("committee_scope").agg(
    hearings=("packageId", "size"), median_words=("n_words", "median"),
    median_turns=("n_turns", "median"), first=("year", "min"), last=("year", "max"))
print(summary.to_string())

print("\n6) hearings per year:")
per_year = back.groupby("year").agg(hearings=("packageId", "size"),
                                    median_words=("n_words", "median"))
print(per_year.to_string())

print("\n7) how the text is structured, which is what the chunker will meet:")
print(f"   paragraph rule    : {back['paragraph_rule'].value_counts().to_dict()}")
print(f"   turns per hearing : median {back['n_turns'].median():,.0f}, "
      f"p90 {back['n_turns'].quantile(0.9):,.0f}, max {int(back['n_turns'].max()):,}")
print(f"   words per hearing : median {back['n_words'].median():,.0f}, "
      f"p90 {back['n_words'].quantile(0.9):,.0f}, max {int(back['n_words'].max()):,}")
single = int((back["n_turns"] <= 1).sum())
print(f"   hearings that came out as a single turn: {single:,}")
qfr = int((back["n_qfr_turns"] > 0).sum())
print(f"   hearings carrying questions for the record: {qfr:,}")

1) 2,352 hearings, packageId unique
2) every row carries a science committee and passes the usability filter
3) every row has turns and clears 8,650 characters
4) all 2,352 qualifying hearings are present

5) shape of the sample:
                 hearings  median_words  median_turns  first  last
committee_scope                                                   
both                    2       43641.5         148.5   2003  2003
house_science         939       22896.0         140.0   1996  2025
senate_commerce      1411       42007.0         154.0   1999  2025

6) hearings per year:
      hearings  median_words
year                        
1996         1       38156.0
1997         1       36236.0
1999        41       31560.0
2000        53       47393.0
2001        77       46247.0
2002        76       38056.5
2003       128       43942.0
2004        81       37144.0
2005        99       41410.0
2006       107       43035.0
2007       151       44724.0
2008        93       42772.0
2009  